# Skenario 2 — Feature Selection dengan Filter Method

**Tujuan**: Menyeleksi fitur yang paling relevan menggunakan metode filter **sebelum** training model, lalu membandingkan hasilnya dengan baseline (Skenario 1).

### Metode Filter yang Digunakan
1. **ANOVA F-test** — mengukur seberapa besar perbedaan rata-rata fitur antar kelas. Skor F tinggi = fitur tersebut mampu membedakan kelas dengan baik. Cocok untuk fitur numerik vs target kategorikal.
2. **Mutual Information** — mengukur dependensi statistik antara fitur dan target (tidak terbatas pada hubungan linear). Menangkap pola non-linear yang mungkin terlewat oleh ANOVA.

### Mengapa Kedua Metode Ini?
- ANOVA F-test menangkap perbedaan **linear** antar kelas
- Mutual Information menangkap hubungan **non-linear**
- Kombinasi keduanya memberikan gambaran lengkap tentang relevansi fitur

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import f_classif, mutual_info_classif, SelectKBest
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('Library berhasil di-import ✅')

---
## 1. Load Data Hasil Preprocessing

In [ ]:
X_train = pd.read_csv('preprocessed_data/X_train.csv')
X_test  = pd.read_csv('preprocessed_data/X_test.csv')
y_train = pd.read_csv('preprocessed_data/y_train.csv').squeeze()
baseline = pd.read_csv('baseline_results.csv', index_col='model')

LABEL_NAMES = {0:'Dewasa Muda (18-35)', 1:'Dewasa (36-53)', 2:'Paruh Baya (54-71)', 3:'Lansia (72-89)'}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'Jumlah fitur awal: {X_train.shape[1]}')

---
## 2. Filter Method 1 — ANOVA F-test

ANOVA F-test menghitung rasio **variasi antar kelas** dibanding **variasi dalam kelas**. Semakin tinggi F-score suatu fitur, semakin baik fitur tersebut dalam membedakan keempat kelompok umur.

Secara intuitif: jika rata-rata kolesterol pada kelompok 'Lansia' sangat berbeda dari 'Dewasa Muda', maka F-score kolesterol akan tinggi.

In [ ]:
# Hitung ANOVA F-score untuk setiap fitur
f_scores, f_pvalues = f_classif(X_train, y_train)

# Buat dataframe ranking
anova_rank = pd.DataFrame({
    'feature': X_train.columns,
    'f_score': f_scores,
    'p_value': f_pvalues
}).sort_values('f_score', ascending=False).reset_index(drop=True)

anova_rank['rank'] = range(1, len(anova_rank)+1)
print('=== Ranking Fitur — ANOVA F-test ===')
print(anova_rank[['rank','feature','f_score','p_value']].to_string(index=False))

In [ ]:
# Visualisasi ranking ANOVA F-score
fig, ax = plt.subplots(figsize=(10, 9))
data = anova_rank.sort_values('f_score', ascending=True)
colors = ['#4C72B0' if p < 0.05 else '#cccccc' for p in data['p_value']]
ax.barh(data['feature'], data['f_score'], color=colors, edgecolor='white')
ax.set_xlabel('F-Score')
ax.set_title('Ranking Fitur — ANOVA F-test', fontsize=14, fontweight='bold')
ax.axvline(x=data['f_score'].median(), color='red', linestyle='--', alpha=0.5, label='Median F-score')
ax.legend()
plt.tight_layout()
plt.savefig('filter_anova_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Biru = signifikan (p<0.05) | Abu-abu = tidak signifikan')

---
## 3. Filter Method 2 — Mutual Information

Mutual Information (MI) mengukur seberapa banyak **informasi** tentang target yang bisa diperoleh dari suatu fitur. Berbeda dengan ANOVA yang hanya menangkap hubungan linear, MI mampu menangkap hubungan **non-linear** apapun.

MI = 0 berarti fitur dan target **independen** (tidak berhubungan sama sekali).

In [ ]:
# Hitung Mutual Information (dengan random_state agar reproducible)
mi_scores = mutual_info_classif(X_train, y_train, random_state=42)

mi_rank = pd.DataFrame({
    'feature': X_train.columns,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False).reset_index(drop=True)

mi_rank['rank'] = range(1, len(mi_rank)+1)
print('=== Ranking Fitur — Mutual Information ===')
print(mi_rank[['rank','feature','mi_score']].to_string(index=False))

In [ ]:
# Visualisasi ranking Mutual Information
fig, ax = plt.subplots(figsize=(10, 9))
data = mi_rank.sort_values('mi_score', ascending=True)
ax.barh(data['feature'], data['mi_score'], color='#55A868', edgecolor='white')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Ranking Fitur — Mutual Information', fontsize=14, fontweight='bold')
ax.axvline(x=data['mi_score'].median(), color='red', linestyle='--', alpha=0.5, label='Median MI')
ax.legend()
plt.tight_layout()
plt.savefig('filter_mi_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Gabungkan Ranking & Pilih Fitur Terbaik

Strategi seleksi: gabungkan ranking dari kedua metode menggunakan **average rank**. Fitur yang konsisten berada di peringkat atas pada kedua metode dianggap paling relevan.

Threshold: pilih fitur yang berada di **top-50% rata-rata ranking** (sekitar 17–18 fitur dari 35).

In [ ]:
# Gabungkan ranking dari kedua metode
combined = pd.DataFrame({'feature': X_train.columns})

# Beri ranking (1 = terbaik)
anova_dict = dict(zip(anova_rank['feature'], anova_rank['rank']))
mi_dict    = dict(zip(mi_rank['feature'], mi_rank['rank']))

combined['anova_rank'] = combined['feature'].map(anova_dict)
combined['mi_rank']    = combined['feature'].map(mi_dict)
combined['avg_rank']   = (combined['anova_rank'] + combined['mi_rank']) / 2
combined = combined.sort_values('avg_rank').reset_index(drop=True)

# Pilih top-50% fitur (threshold = median avg_rank)
threshold = combined['avg_rank'].median()
selected_features = combined[combined['avg_rank'] <= threshold]['feature'].tolist()

print(f'Threshold avg_rank: <= {threshold}')
print(f'Jumlah fitur terpilih: {len(selected_features)} dari {X_train.shape[1]}')
print(f'\n=== Ranking Gabungan ===')
combined['selected'] = combined['avg_rank'] <= threshold
print(combined.to_string(index=False))

In [ ]:
# Visualisasi: fitur terpilih vs tidak terpilih
fig, ax = plt.subplots(figsize=(10, 9))
data = combined.sort_values('avg_rank', ascending=False)
colors = ['#4C72B0' if s else '#dddddd' for s in data['selected']]
ax.barh(data['feature'], data['avg_rank'].max() - data['avg_rank'] + 1, color=colors, edgecolor='white')
ax.set_xlabel('Skor Gabungan (semakin tinggi = semakin penting)')
ax.set_title('Fitur Terpilih (Biru) vs Tidak Terpilih (Abu-abu)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('filter_selected_features.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nFitur terpilih ({len(selected_features)}):')
for i, f in enumerate(selected_features, 1):
    print(f'  {i}. {f}')

### Interpretasi Domain Kesehatan

Fitur-fitur yang terpilih umumnya masuk akal secara domain kesehatan:
- **Bone Density, Vision Sharpness, Hearing Ability** — penanda biologis utama penuaan
- **Cognitive Function** — fungsi kognitif menurun seiring bertambahnya usia
- **Cholesterol, Blood Glucose, BMI** — indikator metabolik yang berubah seiring usia
- **BP_Systolic, BP_Diastolic** — tekanan darah cenderung meningkat pada usia tua

Fitur yang tidak terpilih (misalnya beberapa kolom One-Hot Encoding dari Diet/Family History) memang secara medis **kurang terkait langsung** dengan usia biologis — ini menunjukkan filter method bekerja dengan baik.

---
## 5. Training Model dengan Fitur Terpilih
Gunakan model yang sama dengan baseline (Random Forest, KNN, SVM) agar perbandingan adil.

In [ ]:
# Subset fitur terpilih
X_train_sel = X_train[selected_features]
X_test_sel  = X_test[selected_features]
print(f'Shape setelah seleksi: X_train={X_train_sel.shape}, X_test={X_test_sel.shape}')

# Fungsi evaluasi (sama seperti di baseline)
def evaluate_model(model, X, y, model_name):
    y_pred = cross_val_predict(model, X, y, cv=cv)
    acc  = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred, average='weighted')
    rec  = recall_score(y, y_pred, average='weighted')
    f1   = f1_score(y, y_pred, average='weighted')
    print(f'\n--- {model_name} ---')
    print(f'Accuracy={acc:.4f} | Precision={prec:.4f} | Recall={rec:.4f} | F1={f1:.4f}')
    target_names = [LABEL_NAMES[i] for i in range(4)]
    print(classification_report(y, y_pred, target_names=target_names))
    return {'model': model_name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1_score': f1, 'y_pred': y_pred}

In [ ]:
# Model 1: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_res = evaluate_model(rf, X_train_sel, y_train, 'Random Forest')

In [ ]:
# Model 2: KNN
knn = KNeighborsClassifier(n_neighbors=7, weights='distance', n_jobs=-1)
knn_res = evaluate_model(knn, X_train_sel, y_train, 'KNN (k=7)')

In [ ]:
# Model 3: SVM
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_res = evaluate_model(svm, X_train_sel, y_train, 'SVM (RBF)')

---
## 6. Confusion Matrix — Model dengan Fitur Terpilih

In [ ]:
# Confusion matrix untuk ketiga model
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
labels = [LABEL_NAMES[i] for i in range(4)]

for ax, res in zip(axes, [rf_res, knn_res, svm_res]):
    cm = confusion_matrix(y_train, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels,
                yticklabels=labels, linewidths=0.5, ax=ax)
    ax.set_title(res['model'], fontsize=13, fontweight='bold')
    ax.set_xlabel('Prediksi')
    ax.set_ylabel('Aktual')
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Confusion Matrix — Filter Method (Fitur Terpilih)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('filter_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Perbandingan dengan Baseline (Skenario 1)

In [ ]:
# Tabel perbandingan
filter_results = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'y_pred'}
    for r in [rf_res, knn_res, svm_res]
]).set_index('model')

# Simpan hasil skenario 2
filter_results.to_csv('filter_method_results.csv')

print('=' * 75)
print('PERBANDINGAN: BASELINE (35 fitur) vs FILTER METHOD ({} fitur)'.format(len(selected_features)))
print('=' * 75)

for model_name in baseline.index:
    print(f'\n📌 {model_name}')
    for metric in ['accuracy', 'precision', 'recall', 'f1_score']:
        base_val   = baseline.loc[model_name, metric]
        filter_val = filter_results.loc[model_name, metric]
        diff = filter_val - base_val
        arrow = '🔼' if diff > 0.001 else ('🔽' if diff < -0.001 else '➡️')
        print(f'  {metric:12s}: {base_val:.4f} → {filter_val:.4f} ({diff:+.4f}) {arrow}')

print(f'\nJumlah fitur: 35 → {len(selected_features)} (dikurangi {35-len(selected_features)} fitur)')

In [ ]:
# Visualisasi perbandingan F1-score
fig, ax = plt.subplots(figsize=(10, 5))
models = baseline.index.tolist()
x = np.arange(len(models))
w = 0.35

bars1 = ax.bar(x - w/2, [baseline.loc[m,'f1_score'] for m in models], w, label='Baseline (35 fitur)', color='#4C72B0', alpha=0.85)
bars2 = ax.bar(x + w/2, [filter_results.loc[m,'f1_score'] for m in models], w, label=f'Filter Method ({len(selected_features)} fitur)', color='#55A868', alpha=0.85)

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{bar.get_height():.3f}', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel('F1-Score (weighted)')
ax.set_title('F1-Score: Baseline vs Filter Method', fontsize=14, fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('filter_vs_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Kesimpulan Skenario 2

### Temuan Utama
1. **ANOVA F-test** dan **Mutual Information** memberikan ranking yang relatif konsisten — fitur-fitur biomarker penuaan (bone density, vision, hearing, cognitive function) secara konsisten berada di peringkat atas
2. Dengan mengurangi fitur dari **35 → ~18**, kita mendapatkan model yang lebih **sederhana** dan lebih **interpretable**
3. Fitur yang dieliminasi umumnya adalah kolom One-Hot Encoding dari variabel kategorikal yang memang kurang berkaitan langsung dengan usia biologis

### Analisis Perbandingan
- **Random Forest**: performa cenderung stabil karena algoritma tree-based secara internal sudah melakukan seleksi fitur
- **KNN**: berpotensi **meningkat** karena berkurangnya dimensi mengurangi efek "curse of dimensionality"
- **SVM**: performa bisa berubah tergantung apakah fitur yang dihilangkan memang noise atau informasi berguna

### Catatan untuk Skenario Berikutnya
- Skenario 3 akan mengeksplorasi **Wrapper/Embedded Method** atau **Dimensionality Reduction** (PCA) untuk melihat apakah ada peningkatan lebih lanjut
- Semua hasil dibandingkan terhadap baseline di `baseline_results.csv`